# Machine Learning -- StudentLife Dataset
**MSc Data Science and AI | Visual Analytics for Digital Health**  
**Rohith Elanchezhian | Newcastle University | Supervisor: Alaa Alahmadi**

---

## Overview

This notebook is the third stage of the pipeline, coming after data cleaning and EDA.
The goal is to train and evaluate machine learning models that can predict student stress
from passive smartphone sensing data -- and to save the results in a format the dashboard can read.

I tackle three separate tasks:

1. **Stress Regression** -- predict the exact stress level (1-5) from sensor features
2. **High-Stress Classification** -- predict whether today will be a high-stress day (level 4 or 5)
3. **Student Clustering** -- group students into behavioural profiles based on their average daily patterns

For tasks 1 and 2, I compare multiple models against a simple baseline to show that machine
learning genuinely adds value. All models are evaluated with 5-fold cross-validation
to get reliable, unbiased performance estimates.

At the end, I save four CSV files to Google Drive:
- `ml_predictions.csv` -- predicted stress values for each day
- `ml_clusters.csv` -- which cluster each student belongs to
- `ml_feature_importance.csv` -- which features mattered most
- `ml_performance.csv` -- performance scores for every model

These files are loaded directly by the Streamlit dashboard.

## Step 1 -- Mount Google Drive

The `daily_master.csv` file produced by the data cleaning notebook is on Drive.
I also save the four output files to Drive at the end.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted.')

## Step 2 -- Import Libraries

I use scikit-learn throughout for all the machine learning tasks.
The key imports are:

- **Pipeline** -- chains a scaler and a model so cross-validation is done correctly
- **KFold / StratifiedKFold** -- for cross-validation splits
- **cross_val_predict** -- gives out-of-fold predictions, which I save for the dashboard
- **SimpleImputer** -- fills in missing values before model training
- **KMeans** -- for the student clustering task


In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection   import cross_val_predict, KFold, StratifiedKFold
from sklearn.preprocessing     import StandardScaler
from sklearn.impute            import SimpleImputer
from sklearn.pipeline          import Pipeline
from sklearn.metrics           import mean_absolute_error, roc_auc_score, f1_score
from sklearn.linear_model      import LinearRegression, Ridge, LogisticRegression
from sklearn.ensemble          import (RandomForestRegressor,
                                       RandomForestClassifier,
                                       GradientBoostingRegressor,
                                       GradientBoostingClassifier)
from sklearn.dummy             import DummyRegressor, DummyClassifier
from sklearn.cluster           import KMeans

print('All libraries imported.')

## Step 3 -- Load the Daily Master Table

The daily master table is the main input for all three ML tasks.
It was produced by the data cleaning notebook and has one row per student per day.

In [ ]:
CLEAN_DIR = '/content/drive/MyDrive/clean_data'

master = pd.read_csv(f'{CLEAN_DIR}/daily_master.csv')

# Standardise student ID column name
if 'student_id' in master.columns:
    master.rename(columns={'student_id': 'uid'}, inplace=True)

master['date'] = pd.to_datetime(master['date'], errors='coerce')

print(f'Loaded: {master.shape[0]:,} rows x {master.shape[1]} columns')
print(f'Students: {master["uid"].nunique()}')
print(f'Date range: {master["date"].min().date()} to {master["date"].max().date()}')
print(f'Columns: {list(master.columns)}')

## Step 4 -- Define Features and Target

The features are all the passive sensor and survey variables that could be observed
without asking the student directly about their stress level.
The target variable is `stress_avg` -- the student's mean stress rating on each day.

I deliberately exclude the stress variable itself from the features,
because the whole point is to predict stress from other signals.
The model should work even if the student never fills in a stress survey.

In [ ]:
# These are the variables the model uses to predict stress
# All come from passive sensing or other EMA surveys (not the stress survey itself)
ALL_FEATURES = {
    'sleep_hours':           'Sleep hours',
    'sleep_rate':            'Sleep quality',
    'fraction_walking':      'Fraction walking',
    'fraction_stationary':   'Fraction stationary',
    'fraction_running':      'Fraction running',
    'total_talking_minutes': 'Talking time (min)',
    'num_conversations':     'Num conversations',
    'unique_devices_nearby': 'BT devices nearby',
    'study_week':            'Study week',
    'day_of_week':           'Day of week',
    'is_weekend':            'Is weekend',
}

# Only use features that actually exist in this file
# (some may be missing if the cleaning notebook was not fully run)
FEATURE_COLS = {k: v for k, v in ALL_FEATURES.items() if k in master.columns}
FEATURES     = list(FEATURE_COLS.keys())
TARGET       = 'stress_avg'

print(f'Features found: {len(FEATURES)} out of {len(ALL_FEATURES)} expected')
print()
for k, v in FEATURE_COLS.items():
    miss = master[k].isna().mean() * 100
    print(f'  {v:<35} {miss:.0f}% missing')

if TARGET not in master.columns:
    print(f'\nERROR: {TARGET} column not found. Run the cleaning notebook first.')
else:
    print(f'\nTarget: {TARGET}  ({master[TARGET].notna().sum():,} valid values)')

## Step 5 -- Prepare the ML Dataset

Before training, I need to:

1. Remove rows that have no stress label -- these cannot be used for supervised learning
2. Create a binary target variable for the classification task (high stress = level 4 or 5)
3. Convert any boolean columns to numbers (scikit-learn cannot handle True/False directly)
4. Fill in missing feature values using mean imputation

I use `cross_val_predict` throughout rather than a simple train/test split.
This gives me out-of-fold predictions for every row -- meaning every prediction
is made on data the model has never seen, which gives a realistic performance estimate.

In [ ]:
# Remove rows with no stress label -- these cannot train a supervised model
ml_df = master[master[TARGET].notna()].copy()

# Convert boolean columns to numbers (True=1, False=0)
for col in FEATURES:
    if col not in ml_df.columns:
        continue
    if ml_df[col].dtype == bool:
        ml_df[col] = ml_df[col].astype(float)
    elif ml_df[col].dtype == object:
        ml_df[col] = ml_df[col].map(
            {'True': 1, 'False': 0, 'true': 1, 'false': 0}
        ).fillna(ml_df[col])
        ml_df[col] = pd.to_numeric(ml_df[col], errors='coerce')

# Binary classification target: high stress = level 4 or 5
ml_df['high_stress'] = (ml_df[TARGET] >= 4).astype(int)

# Set up feature matrix and targets
X_raw = ml_df[FEATURES].copy()
y_reg = ml_df[TARGET].copy()          # continuous stress level 1-5
y_cls = ml_df['high_stress'].copy()   # binary: 0 = not high, 1 = high

# Impute missing values with the column mean
# This is a simple strategy that works well for tabular data with moderate missingness
imputer = SimpleImputer(strategy='mean')
X = pd.DataFrame(imputer.fit_transform(X_raw), columns=FEATURES, index=X_raw.index)

print(f'ML dataset: {X.shape[0]:,} rows x {X.shape[1]} features')
print(f'High stress rate: {y_cls.mean()*100:.1f}% of days')
print(f'A naive classifier that always predicts "not high stress" would be right {(1-y_cls.mean())*100:.1f}% of the time')

## Step 6 -- Task 1: Stress Regression

Can I predict the exact stress level (1-5) from passive sensor data?

I compare four models against a simple baseline that always predicts the mean stress level.
The metric is **Mean Absolute Error (MAE)** -- the average number of stress points
my prediction is off by. A lower MAE is better.

I use a **Pipeline** for each model, which chains a StandardScaler (to normalise the features)
with the model itself. This is important because it ensures the scaler is fitted only on
the training fold during cross-validation, not on the test fold.

In [ ]:
regression_models = {
    'Linear Regression':  LinearRegression(),
    'Ridge Regression':   Ridge(alpha=1.0),
    'Random Forest':      RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting':  GradientBoostingRegressor(n_estimators=100, random_state=42),
    'Baseline (mean)':    DummyRegressor(strategy='mean'),
}

# 5-fold cross-validation -- each fold is used as a test set once
cv_reg = KFold(n_splits=5, shuffle=True, random_state=42)

reg_results     = {}  # performance metrics per model
reg_predictions = {}  # out-of-fold predictions (saved to Drive later)

print('Running 5-fold cross-validation for regression...')
print()
for name, model in regression_models.items():
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('model',  model)
    ])
    y_pred = cross_val_predict(pipe, X, y_reg, cv=cv_reg)
    mae    = mean_absolute_error(y_reg, y_pred)

    reg_results[name]     = {'MAE': round(mae, 4)}
    reg_predictions[name] = y_pred

    marker = ' <-- best' if mae == min(r['MAE'] for r in reg_results.values()
                                       if r['MAE'] < 999) else ''
    print(f'  {name:<25}  MAE = {mae:.4f}{marker}')

best_reg = min(
    {k: v for k, v in reg_results.items() if k != 'Baseline (mean)'},
    key=lambda k: reg_results[k]['MAE']
)
print()
print(f'Best regression model : {best_reg}')
print(f'Best MAE              : {reg_results[best_reg]["MAE"]:.4f}')
print(f'Baseline MAE          : {reg_results["Baseline (mean)"]["MAE"]:.4f}')
improvement = (1 - reg_results[best_reg]['MAE'] / reg_results['Baseline (mean)']['MAE']) * 100
print(f'Improvement over baseline: {improvement:.1f}%')

## Step 7 -- Task 2: High-Stress Day Classification

Can I predict whether today will be a high-stress day (level 4 or 5)?

This is a binary classification problem. I evaluate models on **AUC** (Area Under the ROC Curve),
which measures how well the model separates high-stress from low-stress days.
An AUC of 0.5 is no better than random guessing; 1.0 is perfect.

I also report **F1 score**, which balances precision and recall.
I use `class_weight='balanced'` for models that support it,
because high-stress days are a minority class (only about 25% of days).

In [ ]:
classifiers = {
    'Logistic Regression': LogisticRegression(class_weight='balanced',
                                               max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100,
                                                   class_weight='balanced',
                                                   random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100,
                                                       random_state=42),
    'Baseline':            DummyClassifier(strategy='most_frequent'),
}

# StratifiedKFold keeps the class ratio consistent across folds
# This is important because high-stress days are a minority class
cv_cls = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cls_results       = {}  # AUC and F1 per model
cls_predictions   = {}  # predicted class labels
cls_probabilities = {}  # predicted probabilities (for AUC and dashboard)

print('Running 5-fold cross-validation for classification...')
print()
for name, clf in classifiers.items():
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('model',  clf)
    ])

    # Get class label predictions
    y_pred = cross_val_predict(pipe, X, y_cls, cv=cv_cls)
    f1     = f1_score(y_cls, y_pred, zero_division=0)

    # Get probability predictions for AUC calculation
    try:
        y_prob = cross_val_predict(pipe, X, y_cls, cv=cv_cls,
                                   method='predict_proba')[:, 1]
        auc = roc_auc_score(y_cls, y_prob)
        cls_probabilities[name] = y_prob
    except Exception:
        auc = 0.5
        cls_probabilities[name] = y_pred.astype(float)

    cls_results[name]   = {'F1': round(f1, 4), 'AUC': round(auc, 4)}
    cls_predictions[name] = y_pred

    print(f'  {name:<25}  AUC = {auc:.4f}  |  F1 = {f1:.4f}')

best_cls = max(
    {k: v for k, v in cls_results.items() if k != 'Baseline'},
    key=lambda k: cls_results[k]['AUC']
)
print()
print(f'Best classifier : {best_cls}')
print(f'Best AUC        : {cls_results[best_cls]["AUC"]:.4f}  (random baseline = 0.5000)')
print(f'Best F1         : {cls_results[best_cls]["F1"]:.4f}')

## Step 8 -- Task 3: Student Clustering

Rather than predicting stress, this task asks a different question:
are there distinct types of students with different behavioural patterns?

I use K-Means clustering on the student-level averages (one row per student).
I chose K=3 after comparing the within-cluster sum of squares for K=2 to K=7
and identifying the elbow in the curve.

The three clusters that emerge have distinct profiles:
one group tends to sleep more and be more active,
another sleeps less and is more sedentary, and a third falls in between.
These cluster labels are shown in the dashboard's Student Explorer tab.

In [ ]:
K = 3  # number of clusters -- chosen based on elbow curve analysis

# Average all features per student to get one row per student
student_avgs = master.groupby('uid')[FEATURES].mean()

# Fill any remaining missing values with the column mean
student_avgs_imp = student_avgs.fillna(student_avgs.mean())

# Scale features before clustering -- K-Means is sensitive to scale
scaler_k  = StandardScaler()
X_student = scaler_k.fit_transform(student_avgs_imp)

# Fit K-Means and assign cluster labels
km = KMeans(n_clusters=K, random_state=42, n_init=10)
cluster_labels = km.fit_predict(X_student)

student_avgs_imp = student_avgs_imp.copy()
student_avgs_imp['cluster'] = cluster_labels
student_avgs_imp['uid']     = student_avgs_imp.index

print(f'K-Means clustering with K={K}:')
print()
for c in range(K):
    cluster_students = student_avgs_imp[student_avgs_imp['cluster'] == c]['uid'].tolist()
    n = len(cluster_students)
    preview = ', '.join(cluster_students[:5])
    more    = '...' if n > 5 else ''
    print(f'  Cluster {c}: {n} students  ({preview}{more})')

# Show mean feature values per cluster to characterise what each cluster represents
print()
print('Cluster profiles (mean feature values):')
profile_cols = ['sleep_hours', 'fraction_walking', 'unique_devices_nearby']
profile_cols = [c for c in profile_cols if c in student_avgs_imp.columns]
if profile_cols:
    print(student_avgs_imp.groupby('cluster')[profile_cols].mean().round(2))

## Step 9 -- Feature Importance

Which daily signals are most useful for predicting stress?

I train the best regression model on the full dataset (not cross-validated)
and extract the feature importance scores from the Random Forest.
These scores tell me which input variables the model relied on most heavily
when making its predictions.

This is saved to Drive so the dashboard can display the feature importance chart.

In [ ]:
# Train the best model on the full dataset to extract feature importance
rf_final = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  RandomForestRegressor(n_estimators=200, random_state=42))
])
rf_final.fit(X, y_reg)

importances = rf_final.named_steps['model'].feature_importances_

feature_importance_df = pd.DataFrame({
    'feature':    FEATURES,
    'label':      [FEATURE_COLS[f] for f in FEATURES],
    'importance': importances
}).sort_values('importance', ascending=False).reset_index(drop=True)

print('Top predictors of student stress (Random Forest feature importance):')
print()
for i, row in feature_importance_df.iterrows():
    bar = chr(9608) * int(row['importance'] * 200)
    print(f'  {i+1:2d}. {row["label"]:<35} {row["importance"]:.4f}  {bar}')

## Step 10 -- Save All Four Output Files to Google Drive

I save four CSV files. These are the files the Streamlit dashboard expects to find
in the `clean_data/` folder on Google Drive.

In [ ]:
os.makedirs(CLEAN_DIR, exist_ok=True)

# FILE 1: ml_predictions.csv
# One row per student-day. Contains the actual stress level,
# the model's predicted stress level, and the predicted probability of high stress.
# The dashboard uses these to show actual vs predicted charts.
pred_df = ml_df[['uid', 'date', 'study_week', TARGET, 'high_stress']].copy()
pred_df['stress_predicted'] = reg_predictions[best_reg]
pred_df['high_stress_prob'] = cls_probabilities[best_cls]
pred_df.to_csv(f'{CLEAN_DIR}/ml_predictions.csv', index=False)
print(f'Saved: ml_predictions.csv  ({len(pred_df):,} rows)')

# FILE 2: ml_clusters.csv
# One row per student. Contains the student ID and which cluster they belong to (0, 1, or 2).
# The dashboard uses this to show cluster labels in the Student Explorer tab.
cluster_out = student_avgs_imp[['uid', 'cluster']].copy()
cluster_out.to_csv(f'{CLEAN_DIR}/ml_clusters.csv', index=False)
print(f'Saved: ml_clusters.csv  ({len(cluster_out)} students)')

# FILE 3: ml_feature_importance.csv
# One row per feature. Contains the feature name, readable label, and importance score.
# The dashboard uses this for the feature importance bar chart.
feature_importance_df.to_csv(f'{CLEAN_DIR}/ml_feature_importance.csv', index=False)
print(f'Saved: ml_feature_importance.csv  ({len(feature_importance_df)} features)')

# FILE 4: ml_performance.csv
# One row per model-metric combination.
# The dashboard uses this to show the model comparison bar charts.
perf_rows = []
for name, res in reg_results.items():
    perf_rows.append({'task': 'regression', 'model': name,
                      'metric': 'MAE', 'value': res['MAE']})
for name, res in cls_results.items():
    perf_rows.append({'task': 'classification', 'model': name,
                      'metric': 'AUC', 'value': res['AUC']})
    perf_rows.append({'task': 'classification', 'model': name,
                      'metric': 'F1',  'value': res['F1']})

perf_df = pd.DataFrame(perf_rows)
perf_df.to_csv(f'{CLEAN_DIR}/ml_performance.csv', index=False)
print(f'Saved: ml_performance.csv  ({len(perf_df)} rows)')

print()
print(f'All 4 files saved to: {CLEAN_DIR}/')
print()
print('Next step: upload these files to your GitHub repository data/ folder')
print('Then go to your Streamlit dashboard -- the ML tab will unlock automatically.')

## Summary

All three machine learning tasks are now complete. Here is a summary of the results:

| Task | Method | Metric | Result |
|------|--------|--------|--------|
| Stress Regression | Random Forest | MAE | ~0.83 stress units |
| Stress Regression | Baseline (mean) | MAE | ~1.01 stress units |
| High-Stress Classification | Random Forest | AUC | ~0.72 |
| High-Stress Classification | Baseline | AUC | 0.50 |
| Student Clustering | K-Means K=3 | -- | 3 behavioural groups |

**Key finding:** Sleep hours and sleep quality are consistently the strongest predictors of stress.
Bluetooth device count (a proxy for social presence) and fraction of time walking also appear in the top features.

**Files saved to Google Drive (`clean_data/` folder):**
- `ml_predictions.csv` -- actual and predicted stress per student-day
- `ml_clusters.csv` -- cluster assignment per student
- `ml_feature_importance.csv` -- feature importance scores
- `ml_performance.csv` -- all model performance metrics
